# Tokenizer & embeddings — hands-on

Walks the pipeline the model actually consumes: raw Arabic text -> normalized
text -> tokens -> ids -> embedding vectors. The cells import the real package
modules (`tokenizer.py`, `config.py`, `models/embeddings.py`), not a mock.

**Roadmap**
1. Setup — put the package dir on sys.path
2. Load the saved tokenizer
3. Normalization
4. Encode — text to token ids (tokens vs ids)
5. Decode — the round trip
6. Embeddings — ids to vectors (token + position)


## 1. Setup — put the package dir on sys.path

In [ ]:
# Jupyter's cwd is wherever the kernel was started, not necessarily the repo
# root, so we insert the parent of assets/ (i.e. al_haytham_llm/) into sys.path to
# resolve the flat imports (from tokenizer import ..., from config import ...).
# Even `pip install -e .` only provides `al_haytham_llm.tokenizer`; these cells
# import the flat names, so this bootstrap is the honest fix.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0,str(REPO_ROOT))
    
    
print(REPO_ROOT)

## 2. Load the saved tokenizer

In [ ]:
# A tokenizer is trained once over the corpus and saved to assets/tokenizer.json;
# training and inference must reload that exact file or ids would disagree.
#
# vocab_size drives model size: the embedding table is
# (vocab_size x d_model) = 32k x 768 ~= 24.6M parameters.
#   Big vocab   -> shorter sequences, but many rare, under-trained tokens.
#   Small vocab -> cheap tables, but long sequences and more padding.
# Alternatives: char-level (tiny vocab, very long seqs),
#               byte-level (lossless, but noisy tokens).
from tokenizer import Tokenizer
tok = Tokenizer.from_file()
print (f"Vocab Size : {tok.vocab_size}")


## 3. Normalization

In [ ]:
# Normalization rewrites text into one canonical form before splitting into
# tokens — for Arabic it standardizes spelling variants (alef/yeh forms,
# diacritics, tatweel), so differently-spelled text maps to the same ids.
# Why: fewer variants -> the same vocab covers more real text.
# Limitation: distinctions like diacritics are deliberately discarded.
sentence="محمد    عأطف عيسى"
norm_text = tok.normalize(sentence)
print(norm_text)

## 4. Encode — text to token ids

In [ ]:
# encode(): normalize -> split into tokens -> look up integer ids.
# add_special_tokens=True lets the saved post-processor wrap the text with
# framework ids ([PAD], [UNK], [BOS], [EOS], [SEP], language tags [ar]/[en],
# chat roles) — the exact schema the model was pretrained with.
# Alternative: append those ids manually (error-prone, easy to forget one).
encoded = tok.encode(sentence,add_special_tokens=True)
print(encoded)

In [ ]:
# tokenize() stops one step earlier: it returns the token pieces (strings),
# not ids. Encoding = tokenize + id lookup. The model only ever consumes ids.
tokenized = tok.tokenize(sentence)
print(tokenized)

## 5. Decode — the round trip

In [ ]:
# decode() maps ids back to text. skip_special_tokens=False keeps the
# framework tokens visible — a way to sanity-check what encode() produced.
# Why it is not always a perfect round trip: unknown characters become [UNK],
# and normalization runs before splitting, so cosmetics can differ.
decode = tok.decode(encoded,skip_special_tokens=False)
print(decode)

## 6. Embeddings — ids to vectors

Token ids are categorical labels: id arithmetic is meaningless (7 - 3 is not
"token 4"). An embedding is a learned lookup table of shape
`(vocab_size, d_model)` — each id picks a dense vector that the model tunes
during training, so tokens used in similar contexts end up close together.

- **Why not one-hot?** `one_hot(id) @ W == W[id]` — same math, but a one-hot
  vector is 32k wide and mostly zeros; the lookup does it with an index
  instead of a 32k-sized matmul.
- **Position must be injected separately:** attention is
  permutation-equivariant (it treats its input as a set), so `InputEmbedding`
  below adds a learned position vector per time step.
- **Limitation:** the position table is `(seq_len, d_model)` — fixed to the
  trained context window; beyond `seq_len` there is no position left to look
  up (the guard in `embeddings.forward` raises). Modern alternatives:
  sinusoidal (train-free), RoPE / ALiBi (relative, extend better).


In [ ]:
# InputEmbedding = two lookups: token ids -> token table (vocab_size x d_model),
# positions 0..seq_len-1 -> position table (seq_len x d_model), then dropout.
from config import ModelConfig
from models.embeddings import InputEmbedding
from tokenizer import Tokenizer
import torch


tok=Tokenizer.from_file()
my_config = ModelConfig()
embed = InputEmbedding(my_config)
tok = Tokenizer.from_file()


In [ ]:
# A batch is a stack of id sequences: shape (2, S) = (batch, time).
# Why batch: one matmul covers all rows -> far better GPU utilization than
# looping sentence by sentence. Limitation: rows share the time dimension, so
# shorter sentences must be padded with [PAD] (= 0).
sentence_1=" موممد عاكف عيسى"
sentence_2=" موممد عاكف عيسى"


encoded_1 = tok.encode(sentence_1)
encoded_2 = tok.encode(sentence_2)
input_ids=torch.tensor([
    encoded_1,
    encoded_2
])

In [ ]:
# forward(): token vectors + position vectors -> (B, S, D), dropout on top.
# This (batch, time, features) shape flows through the whole transformer;
# attention, FFN and LayerNorm all keep it unchanged.
input_vectors = embed.forward(input_ids=input_ids)
input_vectors.shape
